In [0]:
orders_silver = spark.table("ecommerce_dev.silver.orders")
orders_silver.printSchema()
orders_silver.show(5, truncate=False)

root
 |-- order_id: string (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|bronze_ingested_at     |
+--------------------------------

In [0]:
spark.table("ecommerce_dev.silver.order_items").printSchema()
spark.table("ecommerce_dev.silver.order_payments").printSchema()

root
 |-- order_id: string (nullable = false)
 |-- order_item_id: integer (nullable = false)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)

root
 |-- order_id: string (nullable = false)
 |-- payment_sequential: integer (nullable = false)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- has_invalid_installments: boolean (nullable = true)



In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

orders = spark.table("ecommerce_dev.silver.orders")
dim_customer = spark.table("ecommerce_dev.gold.dim_customer")
dim_date = spark.table("ecommerce_dev.gold.dim_date")

spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.fact_orders (
    order_id STRING NOT NULL,
    customer_key BIGINT,
    order_date_key BIGINT,
    order_status STRING,
    total_items INT,
    total_price DOUBLE,
    total_freight DOUBLE,
    total_payment_value DOUBLE,
    order_purchase_timestamp TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    delivery_days INT,
    is_late BOOLEAN,
    has_missing_delivery_date BOOLEAN
)
USING DELTA
COMMENT 'Gold order fact - grain: one row per order_id. Order-level measures (totals, delivery timing) not captured at line-item grain.'
""")

DataFrame[]

In [0]:
order_items_agg = (
    spark.table("ecommerce_dev.silver.order_items")
    .groupBy("order_id")
    .agg(
        count("*").alias("total_items"),
        sum("price").alias("total_price"),
        sum("freight_value").alias("total_freight")
    )
)

payments_agg = (
    spark.table("ecommerce_dev.silver.order_payments")
    .groupBy("order_id")
    .agg(sum("payment_value").alias("total_payment_value"))
)

fact_orders = (
    orders
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .join(order_items_agg, "order_id", "left")
    .join(payments_agg, "order_id", "left")
    .withColumn("delivery_days",
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("is_late",
                col("order_delivered_customer_date") > col("order_estimated_delivery_date"))
    .withColumn("has_missing_delivery_date",
                (col("order_status") == "delivered") & col("order_delivered_customer_date").isNull())
    .select(
        "order_id", "customer_key", "order_date_key", "order_status",
        "total_items", "total_price", "total_freight", "total_payment_value",
        "order_purchase_timestamp", "order_delivered_customer_date",
        "order_estimated_delivery_date", "delivery_days", "is_late",
        "has_missing_delivery_date"
    )
)

In [0]:
total = fact_orders.count()
nulls = fact_orders.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in ["customer_key", "order_date_key", "total_price"]]
)
print(f"Row count: {total}")
nulls.show()

fact_orders.groupBy("order_status").agg(
    count("*").alias("cnt"),
    sum(col("delivery_days").isNull().cast("int")).alias("null_delivery_days"),
    sum(col("has_missing_delivery_date").cast("int")).alias("missing_delivery_flag")
).show()

Row count: 99441
+------------+--------------+-----------+
|customer_key|order_date_key|total_price|
+------------+--------------+-----------+
|           0|             0|        775|
+------------+--------------+-----------+

+------------+-----+------------------+---------------------+
|order_status|  cnt|null_delivery_days|missing_delivery_flag|
+------------+-----+------------------+---------------------+
|    invoiced|  314|               314|                    0|
|  processing|  301|               301|                    0|
|     shipped| 1107|              1107|                    0|
| unavailable|  609|               609|                    0|
|     created|    5|                 5|                    0|
|    approved|    2|                 2|                    0|
|   delivered|96478|                 8|                    8|
|    canceled|  625|               619|                    0|
+------------+-----+------------------+---------------------+



In [0]:
spark.sql("""
  ALTER TABLE ecommerce_dev.gold.fact_orders
  ADD COLUMN has_missing_delivery_date BOOLEAN
""")

DataFrame[]

In [0]:
target = DeltaTable.forName(spark, "ecommerce_dev.gold.fact_orders")

(target.alias("t")
 .merge(
     fact_orders.alias("s"),
     "t.order_id = s.order_id"
 )
 .whenMatchedUpdate(set={
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "order_status": "s.order_status",
     "total_items": "s.total_items",
     "total_price": "s.total_price",
     "total_freight": "s.total_freight",
     "total_payment_value": "s.total_payment_value",
     "order_purchase_timestamp": "s.order_purchase_timestamp",
     "order_delivered_customer_date": "s.order_delivered_customer_date",
     "order_estimated_delivery_date": "s.order_estimated_delivery_date",
     "delivery_days": "s.delivery_days",
     "is_late": "s.is_late",
     "has_missing_delivery_date": "s.has_missing_delivery_date"
 })
 .whenNotMatchedInsert(values={
     "order_id": "s.order_id",
     "customer_key": "s.customer_key",
     "order_date_key": "s.order_date_key",
     "order_status": "s.order_status",
     "total_items": "s.total_items",
     "total_price": "s.total_price",
     "total_freight": "s.total_freight",
     "total_payment_value": "s.total_payment_value",
     "order_purchase_timestamp": "s.order_purchase_timestamp",
     "order_delivered_customer_date": "s.order_delivered_customer_date",
     "order_estimated_delivery_date": "s.order_estimated_delivery_date",
     "delivery_days": "s.delivery_days",
     "is_late": "s.is_late",
     "has_missing_delivery_date": "s.has_missing_delivery_date"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
  SELECT COUNT(*) AS total,
         SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer,
         SUM(CASE WHEN order_date_key IS NULL THEN 1 ELSE 0 END) AS null_date,
         SUM(CASE WHEN has_missing_delivery_date THEN 1 ELSE 0 END) AS missing_delivery_flag_count
  FROM ecommerce_dev.gold.fact_orders
""").show()

+-----+-------------+---------+---------------------------+
|total|null_customer|null_date|missing_delivery_flag_count|
+-----+-------------+---------+---------------------------+
|99441|            0|        0|                          8|
+-----+-------------+---------+---------------------------+

